In [1]:
import torch
import torch_geometric
from rdkit.Chem import rdmolfiles
from torch_geometric.utils.smiles import from_rdmol
from torch_geometric.data import Dataset, Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, SAGEConv, GATConv, global_mean_pool

import torch.nn.functional as F
from torch import nn

import pandas as pd

from data_processing.common.ids import canonicalize_smiles
from pathlib import Path
import re


/Users/zanechan/anaconda3/envs/pyg_pose/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Workflow:
- Create toy dataset.
    - This dataset should be pathologically easy. 
    - Maybe just like 10 graphs
- Test real model architecture on the toy dataset
- Train that model. 

# Create Toy Dataset
Loading a mini testing dataset of pytorch graphs to test the model training pipeline without lmdb pain

In [2]:
VARIANT_SUFFIX_RE = re.compile(r"^(?P<ligand>.+)-(?P<variant_idx>\d+)$")

def _parse_variant_ligand(variant: str) -> str:
    """Strip the random suffix from `s_lp_Variant` and canonicalize the ligand."""
    variant = str(variant).strip()
    match = VARIANT_SUFFIX_RE.match(variant)
    if not match:
        raise ValueError(
            "Expected s_lp_Variant to look like '{smiles}-{int}', "
            f"got: {variant}"
        )
    return canonicalize_smiles(match.group("ligand"))

def _iter_ligand_mols(sdf_path: Path):
    """Yield `(mol_index, mol)` for valid ligand molecules in a docked SDF."""
    supplier = rdmolfiles.SDMolSupplier(str(sdf_path), removeHs=False)
    for mol_index, mol in enumerate(supplier):
        if mol is None:
            continue
        if not mol.HasProp("s_lp_Variant"):
            continue
        if not mol.HasProp("r_i_docking_score"):
            continue
        if not mol.HasProp("s_i_glide_gridfile"):
            continue
        yield mol_index, mol

def _read_sdf_rows(sdf_path: Path) -> list[dict[str, object]]:
    """Load docked SDF molecules with normalized metadata."""
    rows = []
    resolved_sdf = str(sdf_path.resolve())
    for mol_index, mol in _iter_ligand_mols(sdf_path):
        rows.append(
            {
                "mol_index": mol_index,
                "mol": mol,
                "ligand": _parse_variant_ligand(mol.GetProp("s_lp_Variant")),
                "grid": mol.GetProp("s_i_glide_gridfile").strip(),
                "glide_score": float(mol.GetProp("r_i_docking_score")),
                "source_sdf": resolved_sdf,
            }
        )
    return rows

def rdmol_to_pyg_with_pos(mol):
    """Convert an RDKit molecule into a PyG graph with 3D coordinates."""
    data = from_rdmol(mol)
    conf = mol.GetConformer()
    pos = conf.GetPositions()
    data.pos = torch.tensor(pos, dtype=torch.float)
    data.is_protein_atom = torch.zeros(data.num_nodes, dtype=torch.bool)
    return data

In [3]:
rows = _read_sdf_rows(Path("tests/7BU7_P08588_docked/batch_1_docked.sdf"))
graph = rdmol_to_pyg_with_pos(rows[0]["mol"])
graph

Data(x=[58, 9], edge_index=[2, 120], edge_attr=[120, 3], pos=[58, 3], is_protein_atom=[58])

In [4]:
# prev = rows[0]["mol"].GetNumAtoms()
# for i in range(1000):
#     curr = rows[i]["mol"].GetNumAtoms()
#     if curr != prev:
#         print(i, curr)
#     prev = curr

# ok so we found that at i = 59 prev != curr. There's other examples but this is the first one. 
# our toy dataset will just have 2 graphs. i = 59 and i = 58. 
print(rows[58]["mol"].GetNumAtoms())
print(rows[59]["mol"].GetNumAtoms())

58
57


In [5]:
g1 = rdmol_to_pyg_with_pos(rows[0]["mol"])
g2 = rdmol_to_pyg_with_pos(rows[1]["mol"])

import numpy as np
g1.y = torch.tensor([-10.0])
g2.y = torch.tensor([-1.0])

In [6]:
g1, g2

(Data(x=[58, 9], edge_index=[2, 120], edge_attr=[120, 3], pos=[58, 3], is_protein_atom=[58], y=[1]),
 Data(x=[58, 9], edge_index=[2, 120], edge_attr=[120, 3], pos=[58, 3], is_protein_atom=[58], y=[1]))

In [7]:
print(  torch.all((g1.edge_index == g2.edge_index)),
        torch.all((g1.edge_attr == g2.edge_attr)),
        torch.all((g1.pos == g2.pos)),
        torch.all((g1.x == g2.x))
)


tensor(True) tensor(True) tensor(False) tensor(True)


In [8]:
torch.all(g1.pos == g2.pos)

tensor(False)

In [9]:
g2.pos

tensor([[-74.4618,  18.0761,   2.0355],
        [-73.6774,  18.7021,   0.6545],
        [-73.5984,  20.1566,   0.8333],
        [-74.3587,  18.1378,  -0.5196],
        [-72.0006,  18.0663,   0.8391],
        [-71.2456,  18.5225,   1.9495],
        [-69.9315,  18.0679,   2.1612],
        [-69.0742,  18.5230,   3.6733],
        [-68.1446,  17.4451,   4.0293],
        [-70.0467,  19.0161,   4.6561],
        [-68.1093,  19.8710,   3.2335],
        [-66.6823,  19.7274,   2.9189],
        [-65.8673,  19.4638,   4.2115],
        [-66.3408,  20.2899,   5.3467],
        [-66.0255,  21.7193,   5.2011],
        [-67.0252,  22.6537,   5.8659],
        [-68.2808,  22.4676,   5.2330],
        [-66.5683,  24.1127,   5.6799],
        [-65.2890,  24.2204,   6.2954],
        [-64.8444,  25.4095,   6.8327],
        [-65.6180,  26.5932,   6.9058],
        [-65.1403,  27.6965,   7.6332],
        [-63.8529,  27.6662,   8.1933],
        [-63.0326,  26.5376,   8.0376],
        [-63.5431,  25.4020,   7.3717],


In [10]:
data_list = [g1, g2]
train_loader = DataLoader(data_list, batch_size=2)
val_loader = DataLoader(data_list, batch_size=1)

# Graph Neural Network

In [11]:
# use edge features. WIP
class GCNPoseRegressor(nn.Module):
    def __init__(self, 
                per_atom_dim, 
                per_bond_dim,
                hidden_dim=128,
                num_layers=2):
        super().__init__()

        total_node_dim = per_atom_dim + 3 # atom features, coordinates. Dont't use is_protein_atom for now. We're not using protein graph embeddings. 

        self.conv1 = GATConv(                
                in_channels=per_atom_dim, 
                out_channels=hidden_dim, 
                heads=2, # attention heads
                concat=True, # concat the attention heads
                # negative_slope: float = 0.2, # leaky relu negative slope. 
                # dropout: float = 0.0, # dropout rate — default is 0
                # add_self_loops: bool = True, 
                edge_dim=per_bond_dim
                # fill_value = 'mean', # default is mean for edge attribute of self-loops. Idk if I need to exp with this yet.
                # bias: bool = True, 
                # residual: bool = False # whether or not to add a residual connection. Seems useful but default is False.
                )
        
        self.conv2 = GATConv(
                hidden_dim, 
                hidden_dim,
                heads=2,
                concat=True,
                edge_dim=hidden_dim
                )
        
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, batch):
        x, edge_index, batch_index = batch.x.float(), batch.edge_index, batch.batch
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = global_mean_pool(x, batch_index)
        x = self.regressor(x)
        return x
    

In [12]:
# don't use edge features for now. 
class poseGraphNodeRegressor(nn.Module):
    def __init__(self, 
                per_atom_dim, 
                hidden_dim=128):
        
        super().__init__()
    
        per_atom_dim = per_atom_dim + 3 # atom features, coordinates. 

        self.conv1 = GCNConv(in_channels=per_atom_dim,out_channels=hidden_dim)

        self.conv2 = GCNConv(in_channels=hidden_dim,out_channels=hidden_dim)

        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, batch):
        x, coords, edge_index, batch_index = batch.x.float(), batch.pos, batch.edge_index, batch.batch
        x = torch.cat([x, coords], dim=1)
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = global_mean_pool(x, batch_index)
        x = self.regressor(x)
        return x
        

In [13]:
# per_atom_dim = train_loader.dataset[0].x.shape[1]
# model = poseGraphNodeRegressor(per_atom_dim=per_atom_dim, hidden_dim=128)
# for data in train_loader:
#     print(model(data))

# Train and Evaluate

In [14]:
def train_epoch(model, loader, optimizer, device, epoch):
    model.train()

    total_loss = 0

    for data in loader:
        data = data.to(device)

        optimizer.zero_grad()

        out = model(data)
        y = data.y.view_as(out).float()
        loss = F.mse_loss(out, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * data.num_graphs

    print(f"Epoch {epoch}: Train Loss Item: {loss.item()}, Train Loss Total: {total_loss / len(loader.dataset)}")
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, device, epoch):
    model.eval()

    total_loss = 0

    for data in loader:
        data = data.to(device)

        out = model(data)
        y = data.y.view_as(out).float()
        loss = F.mse_loss(out, y)

        total_loss += loss.item() * data.num_graphs

    print(f"Epoch {epoch}: Val Loss Item: {loss.item()}, Val Loss Total: {total_loss / len(loader.dataset)}")
    return total_loss / len(loader.dataset)

In [15]:
for data in train_loader:
    print(data)
    print(type(data))

DataBatch(x=[116, 9], edge_index=[2, 240], edge_attr=[240, 3], pos=[116, 3], is_protein_atom=[116], y=[2], batch=[116], ptr=[3])
<class 'abc.DataBatch'>


In [16]:
help(torch_geometric.data.Batch)

Help on class Batch in module torch_geometric.data.batch:

class Batch(builtins.object)
 |  Batch(*args: Any, **kwargs: Any) -> Any
 |
 |  A data object describing a batch of graphs as one big (disconnected)
 |  graph.
 |  Inherits from :class:`torch_geometric.data.Data` or
 |  :class:`torch_geometric.data.HeteroData`.
 |  In addition, single graphs can be identified via the assignment vector
 |  :obj:`batch`, which maps each node to its respective graph identifier.
 |
 |  :pyg:`PyG` allows modification to the underlying batching procedure by
 |  overwriting the :meth:`~Data.__inc__` and :meth:`~Data.__cat_dim__`
 |  functionalities.
 |  The :meth:`~Data.__inc__` method defines the incremental count between two
 |  consecutive graph attributes.
 |  By default, :pyg:`PyG` increments attributes by the number of nodes
 |  whenever their attribute names contain the substring :obj:`index`
 |  (for historical reasons), which comes in handy for attributes such as
 |  :obj:`edge_index` or :obj

In [17]:
for data in train_loader:
    print(data.ptr)

tensor([  0,  58, 116])


In [22]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
attn_kwargs = {'dropout': 0.5}
per_atom_dim = train_loader.dataset[0].x.shape[1]
# per_bond_dim = train_loader.dataset[0].edge_attr.shape[1]
model = poseGraphNodeRegressor(per_atom_dim=per_atom_dim, hidden_dim=128).to(device)
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=20,
                            min_lr=0.00001)



In [23]:
for epoch in range(200):
    train_loss = train_epoch(
        model,
        train_loader,
        optimizer,
        device,
        epoch
    )

    val_loss = evaluate(
        model,
        val_loader,
        device,
        epoch
    )

    print(
        f"Epoch {epoch}: "
        f"train={train_loss:.4f}, "
        f"val={val_loss:.4f}"
    )

Epoch 0: Train Loss Item: 51.65030288696289, Train Loss Total: 51.65030288696289
Epoch 0: Val Loss Item: 0.0015980330063030124, Val Loss Total: 39.76407034829026
Epoch 0: train=51.6503, val=39.7641
Epoch 1: Train Loss Item: 39.764076232910156, Train Loss Total: 39.764076232910156
Epoch 1: Val Loss Item: 1.3393582105636597, Val Loss Total: 30.931807935237885
Epoch 1: train=39.7641, val=30.9318
Epoch 2: Train Loss Item: 30.931800842285156, Train Loss Total: 30.931800842285156
Epoch 2: Val Loss Item: 4.706785678863525, Val Loss Total: 25.00891613960266
Epoch 2: train=30.9318, val=25.0089
Epoch 3: Train Loss Item: 25.0089168548584, Train Loss Total: 25.0089168548584
Epoch 3: Val Loss Item: 9.812586784362793, Val Loss Total: 21.2446551322937
Epoch 3: train=25.0089, val=21.2447
Epoch 4: Train Loss Item: 21.24465560913086, Train Loss Total: 21.24465560913086
Epoch 4: Val Loss Item: 16.79450798034668, Val Loss Total: 19.48605251312256
Epoch 4: train=21.2447, val=19.4861
Epoch 5: Train Loss Ite

In [24]:
# g1_batched = g1.clone()
# g1_batched.batch = torch.zeros(g1.num_nodes, dtype=torch.long)
# model.eval()
# with torch.no_grad():
#     pred = model(g1_batched.to(device))

model.eval()
model(g1)

tensor([[-9.9907]], grad_fn=<AddmmBackward0>)

In [25]:
model(g2)

tensor([[-1.0231]], grad_fn=<AddmmBackward0>)

In [26]:
g1, g2

(Data(x=[58, 9], edge_index=[2, 120], edge_attr=[120, 3], pos=[58, 3], is_protein_atom=[58], y=[1]),
 Data(x=[58, 9], edge_index=[2, 120], edge_attr=[120, 3], pos=[58, 3], is_protein_atom=[58], y=[1]))